In [1]:
import random
import torch
import matplotlib.pyplot as plt
import torch.nn.functional as F
%matplotlib inline

words = open('names.txt', 'r').read().splitlines()
chars = sorted(list(set(''.join(words))))
stoi = {s : i + 1 for i, s in enumerate(chars)}
stoi['.'] = 0
itos = {i : s for s, i in stoi.items()}
vocab_size = len(itos)

random.seed(42)
ws = words[:]
random.shuffle(ws)
n1, n2 = int(0.8 * len(ws)), int(0.9 * len(ws))

block_size = 3

def build_dataset(words):
	X, Y = [], []
	for w in words:
		context = [0] * block_size
		for ch in w + '.':
			ix = stoi[ch]
			X.append(context)
			Y.append(ix)
			context = context[1:] + [ix]
	X = torch.tensor(X)
	Y = torch.tensor(Y)
	print(X.shape, Y.shape)
	return X, Y

Xtr, Ytr = build_dataset(ws[:n1])
Xdev,Ydev = build_dataset(ws[n1:n2])
Xte, Yte = build_dataset(ws[n2:])


torch.Size([182625, 3]) torch.Size([182625])
torch.Size([22655, 3]) torch.Size([22655])
torch.Size([22866, 3]) torch.Size([22866])


In [2]:
n_embd = 10
n_hidden = 200

g = torch.Generator().manual_seed(2147483647)
C = torch.randn((vocab_size, n_embd), generator=g)
W1 = torch.randn((block_size * n_embd, n_hidden), generator=g) * (5/3) / ((block_size * n_embd)**0.5)
W2 = torch.randn((n_hidden, vocab_size), generator=g) * 0.1
b2 = torch.randn(vocab_size, generator=g) * 0

bngain = torch.ones((1, n_hidden))
bnbias = torch.zeros((1, n_hidden))
bnmean_running = torch.zeros((1, n_hidden))
bnstd_running = torch.ones((1, n_hidden))

parameters = [C, W1, W2, b2, bngain, bnbias]
for p in parameters:
	p.requires_grad = True

In [3]:
max_steps = 200000
batch_size = 64
lossi = []

for i in range(max_steps):
	ix = torch.randint(0, Xtr.shape[0], (batch_size, ), generator=g)
	Xb, Yb = Xtr[ix], Ytr[ix]

	#forward pass
	emb = C[Xb]
	embcat = emb.view(emb.shape[0], -1)
	hpreact = embcat @ W1
	bnmeani = hpreact.mean(0, keepdim=True)
	bnstdi = hpreact.std(0, keepdim=True)
	hpreact = bngain * (hpreact - bnmeani) / bnstdi + bnbias

	with torch.no_grad():
		bnmean_running = 0.999 * bnmean_running + 0.001 * bnmeani
		bnstd_running = 0.999 * bnstd_running + 0.001 * bnstdi

	h = torch.tanh(hpreact)
	logits = h @ W2 + b2
	loss = F.cross_entropy(logits, Yb)

	#backward pass
	for p in parameters:
		p.grad = None
	loss.backward()

	#update
	lr = 0.1 if i < 100000 else 0.01
	for p in parameters:
		p.data -= lr * p.grad

	lossi.append(loss.log10().item())
	if i % 10000 == 0:
		print(f"{i:7d}/{max_steps:7d} : {loss.item():.4f}")

      0/ 200000 : 3.7435
  10000/ 200000 : 2.2851
  20000/ 200000 : 2.1604
  30000/ 200000 : 2.0604
  40000/ 200000 : 2.4559
  50000/ 200000 : 2.1034
  60000/ 200000 : 1.7511
  70000/ 200000 : 2.3840
  80000/ 200000 : 2.0173
  90000/ 200000 : 2.1456
 100000/ 200000 : 2.3528
 110000/ 200000 : 2.6029
 120000/ 200000 : 2.1116
 130000/ 200000 : 2.2963
 140000/ 200000 : 2.1346
 150000/ 200000 : 1.9824
 160000/ 200000 : 2.0740
 170000/ 200000 : 2.0318
 180000/ 200000 : 2.0126
 190000/ 200000 : 2.0676


In [4]:
@torch.no_grad()
def split_loss(split):
	x, y = {
		'train': (Xtr, Ytr),
		'dev': (Xdev, Ydev),
		'test' : (Xte, Yte)
	}[split]
	emb = C[x]
	embcat = emb.view(emb.shape[0], -1)
	hpreact = embcat @ W1
	hpreact = bngain * (hpreact - bnmean_running) / bnstd_running + bnbias
	h = torch.tanh(hpreact)
	logits = h @ W2 + b2
	loss = F.cross_entropy(logits, y)
	print(split, loss.item())

split_loss('train')
split_loss('dev')

train 2.0458099842071533
dev 2.1003482341766357
